# 05 CosMx cohort (Carver et al., Nature Aging 2026): Figure 3 and Supplementary Table S4
Run 2 of SenNet collection SNT249.TPCD.565: four slides, each with one young (4 mo) and one aged (24 to 25 mo) female C57BL/6 section. Fresh kernel, run C0 to C11 in order.

In [ ]:
# ==========================================================
# C0) CosMx cohort (Carver et al., Nature Aging 2026): setup
# ==========================================================
import re, warnings
from pathlib import Path
import numpy as np, pandas as pd
import scipy.sparse as sp
import anndata as ad
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
from sklearn.cluster import KMeans
import statsmodels.api as sm, statsmodels.formula.api as smf
warnings.filterwarnings("ignore")

import os
# Project folder with data/, tables/, figures/ and final_data/ (see README).
# Default: the parent of the notebooks folder. Override with the environment variable SPP1_PROJECT_DIR.
OUT = Path(os.environ.get("SPP1_PROJECT_DIR", Path.cwd().parent)).resolve()
DATA = OUT / "age_nature_dataset"                       # the four 05..08_Schafer_* slides (run 2)
FIG, TAB = OUT / "figures" / "cosmx", OUT / "tables" / "cosmx"
FIG.mkdir(parents=True, exist_ok=True); TAB.mkdir(parents=True, exist_ok=True)

PX = 0.12028            # CosMx pixel size in µm (Area / Area.um2 in the metadata)
R_NB, R_CLUS = 25, 50   # neighbourhood and cluster radius in µm, as in our Xenium analyses
plt.rcParams.update({"font.family": "sans-serif", "font.size": 7, "axes.spines.top": False,
                     "axes.spines.right": False, "pdf.fonttype": 42, "svg.fonttype": "none"})
C_YOUNG, C_OLD, C_POS, C_AS, C_FIB = "#9aa8dc", "#3b5bdb", "#c2185b", "#3b5bdb", "#9fb3c8"
sig = lambda p: "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "n.s."

def save(fig, name):
    for ext in ["pdf", "png"]:
        fig.savefig(FIG / f"{name}.{ext}", dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
# ==========================================================
# C1) Load the four slides (each slide holds one young and one aged section)
# ==========================================================
slides = sorted(p.name.replace("_exprMat_file.csv", "") for p in DATA.glob("0*_Schafer_*_exprMat_file.csv"))
print(slides)
ads = []
for s in slides:
    ex = pd.read_csv(DATA / f"{s}_exprMat_file.csv")
    me = pd.read_csv(DATA / f"{s}_metadata_file.csv").set_index("cell")
    ex = ex.set_index("cell").drop(columns=["fov", "cell_ID"])
    neg = [c for c in ex.columns if c.startswith("Negative") or c.lower().startswith(("falsecode", "systemcontrol"))]
    genes = [c for c in ex.columns if c not in neg]
    me = me.loc[ex.index]
    obs = pd.DataFrame({"slide": s, "fov": me["fov"].astype(int).values,
                        "x": me["CenterX_global_px"].values * PX, "y": me["CenterY_global_px"].values * PX,
                        "area_um2": me["Area.um2"].values if "Area.um2" in me else me["Area"].values * PX ** 2,
                        "n_neg": ex[neg].sum(axis=1).values}, index=s + "_" + ex.index.astype(str))
    a = ad.AnnData(sp.csr_matrix(ex[genes].values.astype(np.float32)), obs=obs, var=pd.DataFrame(index=genes))
    ads.append(a)
cx = ad.concat(ads, join="inner")
cx.layers["counts"] = cx.X.copy()
cx.obs["n_counts"] = np.asarray(cx.X.sum(1)).ravel()
cx.obs["n_genes"] = np.asarray((cx.X > 0).sum(1)).ravel()
print(cx)
# QC: at least 20 transcripts and 10 genes per cell (as for the public Xenium data)
cx = cx[(cx.obs.n_counts >= 20) & (cx.obs.n_genes >= 10)].copy()
print("cells after QC:", cx.n_obs, "| median transcripts per cell:", cx.obs.n_counts.median())

In [ ]:
# ==========================================================
# C2) Split each slide into its two sections and assign age (young 4 mo, aged 24 to 25 mo)
# ==========================================================
# Two sections per slide: FOVs are split into two spatial groups (k-means on FOV centres).
cx.obs["section"] = ""
for s in slides:
    m = (cx.obs.slide == s).values
    fc = cx.obs[m].groupby("fov")[["x", "y"]].mean()
    lab = KMeans(2, n_init=10, random_state=0).fit_predict(fc.values)
    cx.obs.loc[m, "section"] = [f"{s}|{lab_}" for lab_ in cx.obs.loc[m, "fov"].map(dict(zip(fc.index, lab)))]
# Age by C4b, the most robust age-induced brain transcript: the section with more C4b-detecting cells is aged.
c4b = np.asarray((cx[:, "C4b"].X > 0).todense()).ravel()
age_tab = (pd.DataFrame({"section": cx.obs.section, "c4b": c4b}).groupby("section").c4b.mean() * 100).rename("C4b+ cells (%)").to_frame()
age_tab["slide"] = [i.split("|")[0] for i in age_tab.index]
age_tab["group"] = age_tab.groupby("slide")["C4b+ cells (%)"].transform(lambda v: np.where(v == v.max(), "aged", "young"))
# Animal IDs from the file name, e.g. 05_Schafer_F1_236_YF01: aged = 236, young = YF01
age_tab["mouse"] = [re.search(r"_(\d{3})_", s).group(1) if g == "aged" else re.search(r"_(YF\d+)", s).group(1)
                    for s, g in zip(age_tab.slide, age_tab.group)]
AGE_OVERRIDE = {}       # e.g. {"05_Schafer_F1_236_YF01|0": "aged"} if a check below disagrees
for k, v in AGE_OVERRIDE.items(): age_tab.loc[k, "group"] = v
print(age_tab.round(1).to_string())
cx.obs["group"] = cx.obs.section.map(age_tab.group)
cx.obs["mouse"] = cx.obs.section.map(age_tab.mouse)

fig, axs = plt.subplots(1, len(slides), figsize=(4 * len(slides), 3.6))
for ax, s in zip(np.atleast_1d(axs), slides):
    o = cx.obs[cx.obs.slide == s]
    for g, c in [("young", C_YOUNG), ("aged", C_OLD)]:
        oo = o[o.group == g]; ax.scatter(oo.x, oo.y, s=0.05, c=c, rasterized=True, label=f"{g} ({oo.mouse.iloc[0]})")
    ax.set_aspect("equal"); ax.invert_yaxis(); ax.axis("off"); ax.legend(fontsize=6, markerscale=40, loc="lower left")
    ax.set_title(s, fontsize=7, loc="left")
fig.suptitle("# C2  Section split and age assignment", x=0.01, ha="left", fontsize=8, fontweight="bold")
save(fig, "C2_sections")

In [ ]:
# ==========================================================
# C3) Cell types from canonical marker scores, strict microglia and astrocytes
# ==========================================================
X = cx.layers["counts"]
L = X.multiply(1 / np.maximum(cx.obs.n_counts.values, 1)[:, None]).tocsr() * 100
L.data = np.log1p(L.data)
cx.layers["lognorm"] = L
MARKERS = {"Microglia": ["P2ry12", "Tmem119", "Cx3cr1", "Hexb", "Csf1r", "Sall1"],
           "Astrocyte": ["Aldh1l1", "Gja1", "Slc1a2", "Slc1a3", "Aqp4", "Ntsr2", "Aldoc"],
           "Oligodendrocyte": ["Mbp", "Plp1", "Mog", "Mobp"],
           "OPC": ["Pdgfra", "Gpr17", "Tnr"],
           "Neuron": ["Rbfox3", "Snap25", "Slc17a7", "Gad1", "Gad2"],
           "Endothelial": ["Cldn5", "Flt1", "Pecam1"],
           "Mural": ["Pdgfrb", "Rgs5", "Acta2", "Vtn"],
           "Fibroblast-like": ["Dcn", "Col1a1", "Col1a2"],
           "Ependymal/choroid": ["Foxj1", "Ttr"],
           "Lymphocyte": ["Cd3e", "Ms4a1", "Ptprc"]}
MARKERS = {k: [g for g in v if g in cx.var_names] for k, v in MARKERS.items()}
gi = {g: i for i, g in enumerate(cx.var_names)}
mu = np.asarray(L.mean(0)).ravel(); sd = np.sqrt(np.asarray(L.multiply(L).mean(0)).ravel() - mu ** 2)
scores = pd.DataFrame({ct: np.column_stack([(np.asarray(L[:, gi[g]].todense()).ravel() - mu[gi[g]]) / sd[gi[g]] for g in gs]).mean(1)
                       for ct, gs in MARKERS.items()}, index=cx.obs_names)
cx.obs["cell_type"] = scores.idxmax(axis=1)
# rare types need at least two of their markers, otherwise the cell stays unassigned
for ct_ in ["Lymphocyte", "Fibroblast-like", "Ependymal/choroid", "Mural"]:
    m_ = (cx.obs.cell_type == ct_).values
    nd = np.asarray((X[:, [gi[g] for g in MARKERS[ct_]]] > 0).sum(1)).ravel()
    cx.obs.loc[m_ & (nd < 2), "cell_type"] = "Unassigned"

def n_det(genes, k=2):
    return np.asarray((X[:, [gi[g] for g in genes]] > 0).sum(1)).ravel() >= k
cx.obs["is_mg"] = (cx.obs.cell_type == "Microglia") & n_det(["P2ry12", "Tmem119", "Cx3cr1", "Hexb", "Sall1"])
cx.obs["is_astro"] = (cx.obs.cell_type == "Astrocyte") & n_det(["Aldh1l1", "Gja1", "Slc1a2", "Slc1a3", "Aqp4"])
print(pd.crosstab(cx.obs.cell_type, cx.obs.group))
print("strict microglia:", cx.obs.groupby("mouse").is_mg.sum().to_dict())
print("strict astrocytes:", cx.obs.groupby("mouse").is_astro.sum().to_dict())

# check: marker detection per cell type
chk = [g for gs in MARKERS.values() for g in gs[:3]]
det = pd.DataFrame(np.asarray((X[:, [gi[g] for g in chk]] > 0).todense()), columns=chk, index=cx.obs_names).groupby(cx.obs.cell_type.values).mean() * 100
fig, ax = plt.subplots(figsize=(9, 3.2))
im = ax.imshow(det.values, cmap="Blues", aspect="auto", vmin=0, vmax=100)
ax.set_xticks(range(len(chk))); ax.set_xticklabels(chk, rotation=90, style="italic"); ax.set_yticks(range(len(det))); ax.set_yticklabels(det.index)
plt.colorbar(im, ax=ax, label="cells detecting (%)", shrink=0.8)
ax.set_title("# C3  Marker detection per cell type", loc="left", fontsize=8, fontweight="bold")
save(fig, "C3_marker_check")

In [ ]:
# ==========================================================
# C4) Tissue type: share of oligodendrocytes among the 30 nearest cells (same thresholds as before)
# ==========================================================
cx.obs["f_oligo"] = np.nan
for sec in cx.obs.section.unique():
    m = (cx.obs.section == sec).values; xy = cx.obs.loc[m, ["x", "y"]].values
    _, nn = cKDTree(xy).query(xy, 31)
    cx.obs.loc[m, "f_oligo"] = (cx.obs.loc[m, "cell_type"].values == "Oligodendrocyte")[nn[:, 1:]].mean(1)
cx.obs["ctx_bin"] = pd.cut(cx.obs.f_oligo, [-0.01, 0.2, 0.4, 1.0], labels=["Grey", "Border", "Fibre"]).astype(str)
cx.obs["wm"] = cx.obs.ctx_bin.isin(["Border", "Fibre"])
print((pd.crosstab(cx.obs.mouse, cx.obs.ctx_bin, normalize="index") * 100).round(1))

In [ ]:
# ==========================================================
# C5) Spp1+ microglia: threshold from the CosMx background, more with age, mainly in white matter
# ==========================================================
cx.obs["Spp1_n"] = np.asarray(X[:, gi["Spp1"]].todense()).ravel()
# CosMx detects some Spp1 in every cell type (spillover and background). The threshold is the lowest count
# at which at most 1% of non-microglial cells pass, so that Spp1+ marks microglia that truly express Spp1.
nonmg = ~cx.obs.is_mg & (cx.obs.cell_type != "Microglia")
thr_tab = pd.DataFrame({k: [100 * (cx.obs.Spp1_n[nonmg] >= k).mean(), 100 * (cx.obs.Spp1_n[cx.obs.is_mg] >= k).mean()] for k in range(1, 6)},
                       index=["non-microglia (%)", "microglia (%)"]).round(2)
print(thr_tab)
SPP1_MIN = int(next(k for k in range(1, 10) if (cx.obs.Spp1_n[nonmg] >= k).mean() <= 0.01))
print("Spp1 threshold (transcripts):", SPP1_MIN)
cx.obs["spp1"] = ((cx.obs.Spp1_n >= SPP1_MIN) & cx.obs.is_mg).astype(int)

mg = cx.obs[cx.obs.is_mg].copy()
mg["aged"] = (mg.group == "aged").astype(int)
per_mouse = (mg.groupby(["slide", "mouse", "group", "wm"]).spp1.agg(["sum", "size"])
             .assign(pct=lambda d: 100 * d["sum"] / d["size"]).reset_index())
per_mouse["tissue"] = np.where(per_mouse.wm, "in and around white matter", "grey matter")
print(per_mouse.round(2).to_string(index=False))
per_mouse.to_csv(TAB / "C5_spp1_per_mouse.csv", index=False)
# Age effect with mouse as the unit: binomial regression with tissue type, standard errors clustered by mouse
f_age = smf.glm("spp1 ~ aged * wm", data=mg.assign(wm=mg.wm.astype(int)), family=sm.families.Binomial()).fit(
    cov_type="cluster", cov_kwds={"groups": mg.mouse.astype("category").cat.codes})
print(f_age.summary().tables[1])
# Paired by slide (each slide holds one young and one aged mouse)
from scipy.stats import ttest_rel
for t in ["in and around white matter", "grey matter"]:
    w = per_mouse[per_mouse.tissue == t].pivot(index="slide", columns="group", values="pct")
    print(t, "| young", w.young.round(2).tolist(), "| aged", w.aged.round(2).tolist(), "| paired t p =", round(ttest_rel(w.aged, w.young).pvalue, 4))

# Clustering per mouse: share of Spp1+ microglia with another Spp1+ microglia within 50 µm vs shuffles within tissue type
def clustering(o, nperm=1000, seed=0):
    xy, lab, ctx = o[["x", "y"]].values, o.spp1.values.astype(bool), o.ctx_bin.values
    if lab.sum() < 5: return np.nan, np.nan, np.nan
    pairs = cKDTree(xy).query_ball_point(xy, R_CLUS)
    def score(l): return np.mean([sum(l[j] for j in pairs[i] if j != i) > 0 for i in np.where(l)[0]])
    obs_ = score(lab); rng = np.random.default_rng(seed); null = []
    for _ in range(nperm):
        l = lab.copy()
        for b in np.unique(ctx):
            ib = np.where(ctx == b)[0]; l[ib] = rng.permutation(l[ib])
        null.append(score(l))
    null = np.array(null); return obs_, null.mean(), (np.sum(null >= obs_) + 1) / (nperm + 1)
clus = pd.DataFrame([(mo, *clustering(mg[mg.mouse == mo], nperm=500)) for mo in mg.mouse.unique()],
                    columns=["mouse", "observed", "expected", "p"]).merge(age_tab[["mouse", "group"]], on="mouse")
print(clus.round(3).to_string(index=False)); clus.to_csv(TAB / "C5_clustering_per_mouse.csv", index=False)

In [ ]:
# ==========================================================
# C6) Fewer astrocyte neighbours: per mouse, Spp1+ vs Spp1- microglia of the same tissue type
# ==========================================================
mg["n_astro25"] = 0
for sec in cx.obs.section.unique():
    o = cx.obs[cx.obs.section == sec]
    t_as = cKDTree(o.loc[o.is_astro, ["x", "y"]].values)
    idx = mg.index[mg.section == sec]
    mg.loc[idx, "n_astro25"] = t_as.query_ball_point(mg.loc[idx, ["x", "y"]].values, R_NB, return_length=True)
rows = []
for mo, d in mg.groupby("mouse"):
    r = {"mouse": mo, "group": d.group.iloc[0], "n_spp1_pos": int(d.spp1.sum())}
    for t, dd in [("wm", d[d.wm]), ("all", d)]:
        c = dd.n_astro25 >= 1
        r[f"contact_pos_{t}"] = 100 * c[dd.spp1 == 1].mean() if dd.spp1.sum() else np.nan
        r[f"contact_neg_{t}"] = 100 * c[dd.spp1 == 0].mean()
    if d.spp1.sum() >= 10:
        f = smf.glm("spp1 ~ n_astro25 + C(ctx_bin)", data=d, family=sm.families.Binomial()).fit(cov_type="HC1")
        r.update(OR=np.exp(f.params["n_astro25"]), lo=np.exp(f.conf_int().loc["n_astro25", 0]),
                 hi=np.exp(f.conf_int().loc["n_astro25", 1]), p=f.pvalues["n_astro25"])
    rows.append(r)
astro = pd.DataFrame(rows).sort_values(["group", "mouse"]); print(astro.round(3).to_string(index=False))
astro.to_csv(TAB / "C6_astro_neighbours_per_mouse.csv", index=False)
# Pooled over aged mice, mouse as fixed effect, standard errors clustered by mouse
ma = mg[mg.group == "aged"]
if ma.mouse.nunique() > 1:
  f_pool = smf.glm("spp1 ~ n_astro25 + C(ctx_bin) + C(mouse)", data=ma, family=sm.families.Binomial()).fit(
      cov_type="cluster", cov_kwds={"groups": ma.mouse.astype("category").cat.codes})
  print("aged pooled: OR per astrocyte neighbour", np.exp(f_pool.params["n_astro25"]).round(3),
        np.exp(f_pool.conf_int().loc["n_astro25"]).round(3).values, "p =", f_pool.pvalues["n_astro25"])
# Composition control (as in our cohort): share of astrocytes and oligodendrocytes and number of cells within 25 µm
comp = []
for sec in cx.obs.section.unique():
    o = cx.obs[cx.obs.section == sec]; t_all = cKDTree(o[["x", "y"]].values)
    idx = mg.index[mg.section == sec]
    nb = t_all.query_ball_point(mg.loc[idx, ["x", "y"]].values, R_NB)
    ct_o = o.cell_type.values; ia = o.is_astro.values
    for i, n_ in zip(idx, nb):
        n_ = [j for j in n_ if o.index[j] != i]
        comp.append({"cell": i, "n_cells": len(n_), "f_astro": ia[n_].mean() if n_ else np.nan,
                     "f_oligo_nb": (ct_o[n_] == "Oligodendrocyte").mean() if n_ else np.nan})
mg = mg.join(pd.DataFrame(comp).set_index("cell"))
print((mg[mg.group == "aged"].groupby(["mouse", "spp1"])[["n_cells", "f_astro", "f_oligo_nb"]].mean()).round(3))

In [ ]:
# ==========================================================
# C7) Are the astrocytes next to Spp1+ microglia reactive? (per mouse, same tissue type)
# ==========================================================
GA = [g for g in ["Gfap", "Serpina3n", "C4b", "Vim", "Cd44", "Lgals3", "Aqp4", "Aldh1l1", "Gja1", "Slc1a2", "Ntsr2", "Aldoc"] if g in gi]
def astro_near(o, nperm=2000, seed=0):
    xy = o[["x", "y"]].values; ia, im = o.is_astro.values, o.is_mg.values; ip = im & (o.spp1.values == 1)
    near_p = cKDTree(xy[ip]).query_ball_point(xy[ia], R_NB, return_length=True) > 0
    far_p = cKDTree(xy[ip]).query_ball_point(xy[ia], 50, return_length=True) == 0
    near_n = cKDTree(xy[im & ~ip]).query_ball_point(xy[ia], R_NB, return_length=True) > 0
    keep = near_p | (near_n & far_p); lab = near_p[keep]
    rows_ = np.where(ia)[0][keep]; strat = o.ctx_bin.values[rows_]
    E = np.asarray(cx[o.index[rows_], GA].layers["lognorm"].todense())
    rng = np.random.default_rng(seed); perms = np.empty((nperm, len(lab)), bool)
    for i in range(nperm):
        l = lab.copy()
        for b in np.unique(strat):
            ib = np.where(strat == b)[0]; l[ib] = rng.permutation(l[ib])
        perms[i] = l
    out = []
    for j, g in enumerate(GA):
        e = E[:, j]; d = e[lab].mean() - e[~lab].mean()
        null = (perms @ e) / lab.sum() - ((~perms) @ e) / (~lab).sum()
        out.append({"gene": g, "log2FC": np.log2(np.expm1(e[lab].mean()) / max(np.expm1(e[~lab].mean()), 1e-9)),
                    "det_pos": 100 * (e[lab] > 0).mean(), "det_neg": 100 * (e[~lab] > 0).mean(),
                    "p": (np.sum(np.abs(null) >= abs(d)) + 1) / (nperm + 1), "n_pos": int(lab.sum()), "n_neg": int((~lab).sum())})
    return pd.DataFrame(out)
react = []
for mo in sorted(cx.obs.mouse.unique()):
    o = cx.obs[cx.obs.mouse == mo].copy(); o["spp1"] = 0
    o.loc[mg.index[(mg.mouse == mo)], "spp1"] = mg.loc[mg.mouse == mo, "spp1"]
    if o.spp1.sum() < 5: continue
    react.append(astro_near(o, nperm=1000).assign(mouse=mo, group=o.group.iloc[0]))
react = pd.concat(react); react.to_csv(TAB / "C7_astro_genes_per_mouse.csv", index=False)
print(react.pivot_table(index="gene", columns="mouse", values="log2FC").round(2))
print(react.pivot_table(index="gene", columns="mouse", values="p").round(3))

In [ ]:
# ==========================================================
# C8) Identity of Spp1+ microglia (aged mice, depth-matched to 100 transcripts)
# ==========================================================
GI = [g for g in ["P2ry12", "Tmem119", "Cx3cr1", "Sall1", "Hexb", "Csf1r", "Trem2", "Tyrobp", "Apoe", "Cd9", "Clec7a", "Cst7", "Lpl",
                  "Itgax", "Gpnmb", "Cd74", "B2m", "Lgals3", "Ctsb", "Cd63", "Ctss", "Cdkn1a", "Cdkn2a", "Il1a", "Ccl3", "Ccl4"] if g in gi]
rng = np.random.default_rng(0)
ma = mg[mg.group == "aged"]
Cm = np.asarray(cx[ma.index, GI].layers["counts"].todense()).astype(int)
tot = ma.n_counts.values.astype(int)
p_keep = np.minimum(1, 100 / np.maximum(tot, 1))[:, None]
Cd = rng.binomial(Cm, p_keep); ok = tot >= 100
ident = pd.DataFrame({"Spp1+": (Cd[ok & (ma.spp1.values == 1)] > 0).mean(0) * 100,
                      "Spp1-": (Cd[ok & (ma.spp1.values == 0)] > 0).mean(0) * 100}, index=GI)
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests
pv = [fisher_exact([[int((Cd[ok & (ma.spp1.values == 1), j] > 0).sum()), int((Cd[ok & (ma.spp1.values == 1), j] == 0).sum())],
                    [int((Cd[ok & (ma.spp1.values == 0), j] > 0).sum()), int((Cd[ok & (ma.spp1.values == 0), j] == 0).sum())]])[1] for j in range(len(GI))]
ident["diff"] = ident["Spp1+"] - ident["Spp1-"]; ident["q"] = multipletests(pv, method="fdr_bh")[1]
print(ident.round(3).sort_values("diff").to_string()); ident.to_csv(TAB / "C8_identity_aged.csv")

In [ ]:
# ==========================================================
# C9) Astrocyte niche without transcript files: three complementary measures
# ==========================================================
# (a) Astrocyte RNA in the neighbourhood: homeostatic astrocyte transcripts in ALL non-microglial cells
#     whose centre lies 5 to 25 µm from the microglia (captures astrocyte processes that were segmented into
#     neighbouring cells, the closest proxy for the transcript-level analysis)
# (b) Distance to the nearest astrocyte cell body
# (c) Astrocyte neighbours across radii (10 to 50 µm), to check that the result does not depend on one radius
# (d) GFAP protein (immunofluorescence channel of CosMx) in neighbouring cells, protein-level reactivity
globals().pop("C", None)                          # C8 used the name C, which hides patsy's C() in the formulas below
GH = [g for g in ["Aldh1l1", "Gja1", "Ntsr2", "Slc1a2", "Slc1a3"] if g in gi]
H = np.asarray(cx[:, GH].layers["counts"].sum(1)).ravel()
mean_gfap = {}
for s in slides:                                   # GFAP immunofluorescence per cell from the metadata
    me = pd.read_csv(DATA / f"{s}_metadata_file.csv", usecols=["cell", "Mean.GFAP"])
    mean_gfap.update(dict(zip(s + "_" + me["cell"].astype(str), me["Mean.GFAP"])))
cx.obs["gfap_if"] = cx.obs_names.map(mean_gfap).astype(float)

RADII = [10, 15, 20, 25, 35, 50]
out = []
for sec in cx.obs.section.unique():
    o = cx.obs[cx.obs.section == sec]
    xy = o[["x", "y"]].values
    nonmg = ~o.is_mg.values
    h = H[cx.obs_names.get_indexer(o.index)]
    t_all = cKDTree(xy[nonmg]); t_as = cKDTree(xy[o.is_astro.values])
    im = np.where(o.is_mg.values)[0]; xm = xy[im]
    r = pd.DataFrame(index=o.index[im])
    nb25 = t_all.query_ball_point(xm, 25); nb5 = t_all.query_ball_point(xm, 5)
    hn, gf = h[nonmg], o.gfap_if.values[nonmg]
    r["astro_rna"] = [hn[a].sum() - hn[b].sum() for a, b in zip(nb25, nb5)]
    r["n_cells25"] = [len(a) for a in nb25]
    r["gfap_if_nb"] = [gf[a].mean() if len(a) else np.nan for a in nb25]
    r["d_astro"] = t_as.query(xm, 1)[0]
    for R in RADII:
        r[f"n_astro_{R}"] = t_as.query_ball_point(xm, R, return_length=True)
    out.append(r)
nb = pd.concat(out).join(cx.obs[["mouse", "group", "ctx_bin", "wm", "spp1"]])
nb.to_csv(TAB / "C9_niche_measures_per_microglia.csv")

def fit(d, form, fam):
    f = smf.glm(form, data=d, family=fam).fit(cov_type="HC1"); return np.exp(f.params["spp1"]), f.pvalues["spp1"]
res = []
for mo, d in nb.groupby("mouse"):
    if d.spp1.sum() < 10: continue
    r = {"mouse": mo, "group": d.group.iloc[0], "n_pos": int(d.spp1.sum())}
    r["RR_astro_rna"], r["p_astro_rna"] = fit(d, "astro_rna ~ spp1 + C(ctx_bin) + n_cells25", sm.families.Poisson())
    r["d_astro_pos"], r["d_astro_neg"] = d.loc[d.spp1 == 1, "d_astro"].median(), d.loc[d.spp1 == 0, "d_astro"].median()
    f = smf.ols("np.log(d_astro + 1) ~ spp1 + C(ctx_bin)", data=d).fit(cov_type="HC1")
    r["ratio_d_astro"], r["p_d_astro"] = np.exp(f.params["spp1"]), f.pvalues["spp1"]
    for R in RADII:
        r[f"RR_n_astro_{R}"] = fit(d, f"n_astro_{R} ~ spp1 + C(ctx_bin)", sm.families.Poisson())[0]
    f = smf.ols("np.log1p(gfap_if_nb) ~ spp1 + C(ctx_bin)", data=d.dropna(subset=["gfap_if_nb"])).fit(cov_type="HC1")
    r["ratio_gfap_if"], r["p_gfap_if"] = np.exp(f.params["spp1"]), f.pvalues["spp1"]
    res.append(r)
res = pd.DataFrame(res); res.to_csv(TAB / "C9_niche_per_mouse.csv", index=False)
print(res[["mouse", "group", "n_pos", "RR_astro_rna", "p_astro_rna", "d_astro_pos", "d_astro_neg", "ratio_d_astro", "p_d_astro",
           "ratio_gfap_if", "p_gfap_if"]].round(3).to_string(index=False))
print(res[["mouse"] + [f"RR_n_astro_{R}" for R in RADII]].round(2).to_string(index=False))

In [ ]:
# ==========================================================
# C10) Do very old brains become amyloid-like? Widespread vs local astrocyte reactivity, per mouse
# ==========================================================
# Amyloid pattern (10x TgCRND8): astrocytes far from Spp1+ microglia are already reactive, the local excess is small.
# Ageing pattern (13.4 mo WT): distant astrocytes are quiet, the local excess is large.
GRx = [g for g in ["Gfap", "Serpina3n", "C4b", "Vim", "Cd44"] if g in gi]
rows = []
for mo in sorted(cx.obs.mouse.unique()):
    o = cx.obs[cx.obs.mouse == mo]; xy = o[["x", "y"]].values
    ip = (o.is_mg & (o.spp1 == 1)).values; ia = o.is_astro.values
    if ip.sum() == 0: d_p = np.full(ia.sum(), np.inf)
    else: d_p = cKDTree(xy[ip]).query(xy[ia], 1)[0]
    E = pd.DataFrame(np.asarray(cx[o.index[ia], GRx].layers["lognorm"].todense()), columns=GRx)
    D = pd.DataFrame(np.asarray((cx[o.index[ia], GRx].layers["counts"] > 0).todense()), columns=GRx)
    near, far = d_p <= R_NB, d_p > 50
    wm_a = o.wm.values[ia]
    for g in GRx:
        rows.append({"mouse": mo, "group": o.group.iloc[0], "gene": g,
                     "det_far_wm": 100 * D.loc[far & wm_a, g].mean(), "det_far_grey": 100 * D.loc[far & ~wm_a, g].mean(),
                     "det_near": 100 * D.loc[near, g].mean() if near.any() else np.nan,
                     "log2FC_near_vs_far_wm": np.log2(np.expm1(E.loc[near, g].mean()) / max(np.expm1(E.loc[far & wm_a, g].mean()), 1e-9)) if near.any() else np.nan})
glob = pd.DataFrame(rows); glob.to_csv(TAB / "C10_widespread_vs_local.csv", index=False)
print(glob.pivot_table(index="gene", columns=["group", "mouse"], values="det_far_wm").round(1))
print(glob.pivot_table(index="gene", columns=["group", "mouse"], values="log2FC_near_vs_far_wm").round(2))
# Microglia: is the Spp1+ state at 25 months Trem2-high like under amyloid? (detection in Spp1+ microglia, depth-matched, see C8)
print(ident.loc[[g for g in ["Trem2", "Tyrobp", "Gpnmb", "Cd74", "Itgax", "Lpl", "Clec7a", "Cst7"] if g in ident.index], ["Spp1+", "Spp1-", "q"]].round(3))

In [ ]:
# ==========================================================
# C11) Figure 3: old age (CosMx, 4 young vs 4 aged mice) + Supplementary Table S4 (source data)
# ==========================================================
import matplotlib.gridspec as gridspec
from matplotlib.patches import Circle, Patch
from matplotlib.colors import LinearSegmentedColormap, to_rgba
from openpyxl.styles import Font
globals().pop("C", None)
MM = 1 / 25.4
C_MG, C_WM, C_BOR, C_BG = "#e0a100", "#bfcde0", "#dde4ee", "#ececec"
CM_DOT = LinearSegmentedColormap.from_list("dot", ["#f1f3fb", "#9aa8dc", "#2b3f9e"])
CM_DIV = LinearSegmentedColormap.from_list("div", ["#3b5bdb", "#f7f7f7", "#c2185b"])
AGE_MO = {"236": 24, "237": 24, "239": 24, "243": 25}            # SenNet source metadata, young mice (YF) 4 months
LBLM = {r.mouse: f"{r.mouse}, {AGE_MO.get(r.mouse, 4)} mo" for r in age_tab.itertuples()}
OLDLAB = "24 to 25 mo"
PAIR = slides[-1]                                   # slide shown in A and B (one young and one aged section)
mo_y, mo_a = [age_tab[(age_tab.slide == PAIR) & (age_tab.group == g)].mouse.iloc[0] for g in ["young", "aged"]]

def tissue_img(o, res=20):
    xb = np.arange(o.x.min(), o.x.max() + res, res); yb = np.arange(o.y.min(), o.y.max() + res, res)
    Ha, _, _ = np.histogram2d(o.x, o.y, bins=[xb, yb]); Hw, _, _ = np.histogram2d(o.x[o.wm], o.y[o.wm], bins=[xb, yb])
    img = np.zeros(Ha.shape + (4,)); img[Ha > 0] = to_rgba(C_BG); img[(Ha > 0) & (Hw / np.maximum(Ha, 1) >= 0.5)] = to_rgba(C_WM)
    return img.transpose(1, 0, 2), (xb[0], xb[-1], yb[0], yb[-1])

fig = plt.figure(figsize=(180 * MM, 205 * MM))
gs = gridspec.GridSpec(3, 12, figure=fig, height_ratios=[1.0, 0.9, 0.95], hspace=0.6, wspace=2.6,
                       left=0.08, right=0.96, top=0.96, bottom=0.06)

# ================= A: one slide, young and aged section at a common scale
gA = gs[0, 0:8].subgridspec(1, 2, wspace=0.05)
secs = [(mo_y, cx.obs[cx.obs.mouse == mo_y]), (mo_a, cx.obs[cx.obs.mouse == mo_a])]
span = max(max(o.x.max() - o.x.min(), o.y.max() - o.y.min()) for _, o in secs) * 1.03
for k, (mo, o) in enumerate(secs):
    ax = fig.add_subplot(gA[k]); img, ext = tissue_img(o)
    ax.imshow(img, extent=ext, origin="lower", interpolation="nearest")
    p_ = o[o.is_mg & (o.spp1 == 1)]
    ax.scatter(p_.x, p_.y, s=7, c=C_POS, lw=0.25, edgecolor="white", zorder=5)
    cxs, cys = (o.x.min() + o.x.max()) / 2, (o.y.min() + o.y.max()) / 2
    ax.set_xlim(cxs - span / 2, cxs + span / 2); ax.set_ylim(cys - span / 2, cys + span / 2)
    ax.set_aspect("equal"); ax.axis("off")
    ax.plot([cxs + span / 2 - 1100, cxs + span / 2 - 100], [cys - span / 2 + 120] * 2, color="#222222", lw=1.1)
    ax.text(cxs + span / 2 - 600, cys - span / 2 + 230, "1 mm", ha="center", fontsize=5)
    ax.set_title(("# 3A  " if k == 0 else "") + LBLM[mo], loc="left", fontsize=7, fontweight="bold" if k == 0 else "normal")
    if k == 1:
        for lab_, c_ in [("White matter", C_WM), ("Spp1+ microglia", C_POS)]:
            ax.scatter([], [], s=12, c=c_, marker="s" if c_ == C_WM else "o", label=lab_)
        ax.legend(loc="upper right", fontsize=5.2, frameon=False)

# ================= B: zoom on a cluster in the aged section
axB = fig.add_subplot(gs[0, 8:12])
oa = cx.obs[cx.obs.mouse == mo_a]
pf = oa[oa.is_mg & (oa.spp1 == 1) & oa.wm]
pxy = pf[["x", "y"]].values; cxz, cyz = pxy[cKDTree(pxy).query_ball_point(pxy, 100, return_length=True).argmax()]
WZ = 220
z = oa[oa.x.between(cxz - WZ / 2, cxz + WZ / 2) & oa.y.between(cyz - WZ / 2, cyz + WZ / 2)]
axB.scatter(z.x, z.y, s=4, c=np.where(z.cell_type == "Oligodendrocyte", C_FIB, "#e6e6e6"), lw=0)
za = z[z.is_astro]; axB.scatter(za.x, za.y, s=11, c=C_AS, lw=0, zorder=3)
zm = z[z.is_mg]; axB.scatter(zm.x, zm.y, s=11, c=C_MG, lw=0, zorder=4)
zp = zm[zm.spp1 == 1]; axB.scatter(zp.x, zp.y, s=26, c=C_POS, lw=0.5, edgecolor="white", zorder=5)
for x_, y_ in zp[["x", "y"]].values:
    axB.add_patch(Circle((x_, y_), R_NB, fill=False, ec="#888888", lw=0.4, ls=(0, (2, 2)), zorder=2))
axB.set_xlim(cxz - WZ / 2, cxz + WZ / 2); axB.set_ylim(cyz - WZ / 2, cyz + WZ / 2); axB.set_aspect("equal")
axB.set_xticks([]); axB.set_yticks([])
for s_ in axB.spines.values(): s_.set_visible(True); s_.set_linewidth(0.6)
axB.plot([cxz + WZ / 2 - 60, cxz + WZ / 2 - 10], [cyz - WZ / 2 + 12] * 2, color="#222222", lw=1.1)
axB.text(cxz + WZ / 2 - 35, cyz - WZ / 2 + 20, "50 µm", ha="center", fontsize=5)
for lab_, c_ in [("Oligodendrocyte", C_FIB), ("Astrocyte", C_AS), ("Microglia", C_MG), ("Spp1+ microglia", C_POS)]:
    axB.scatter([], [], s=10, c=c_, label=lab_)
axB.legend(loc="upper left", bbox_to_anchor=(-0.05, -0.02), ncol=2, fontsize=4.9, handletextpad=0.2, columnspacing=0.4, frameon=False)
axB.set_title(f"# 3B  Spp1+ cluster, {LBLM[mo_a]}", loc="left", fontsize=7, fontweight="bold")

# ================= C: Spp1+ share per mouse, paired by slide
axC = fig.add_subplot(gs[1, 0:4])
for k, t in enumerate(["in and around white matter", "grey matter"]):
    w = per_mouse[per_mouse.tissue == t].pivot(index="slide", columns="group", values="pct")
    for sl, r_ in w.iterrows():
        axC.plot([k * 2.4, k * 2.4 + 1], [r_.young, r_.aged], color="#bbbbbb", lw=0.7, zorder=1)
    axC.scatter(np.full(len(w), k * 2.4), w.young, s=18, c=C_YOUNG, zorder=3)
    axC.scatter(np.full(len(w), k * 2.4 + 1), w.aged, s=18, c=C_OLD, zorder=3)
    p_ = ttest_rel(w.aged, w.young).pvalue
    top = max(w.aged.max(), w.young.max()) + 1.5
    axC.plot([k * 2.4, k * 2.4, k * 2.4 + 1, k * 2.4 + 1], [top - 0.5, top, top, top - 0.5], color="#333333", lw=0.6)
    axC.text(k * 2.4 + 0.5, top + 0.3, sig(p_), ha="center", fontsize=6)
axC.set_xticks([0, 1, 2.4, 3.4]); axC.set_xticklabels(["4 mo", "24 to\n25 mo", "4 mo", "24 to\n25 mo"], fontsize=5.6)
axC.text(0.5, -0.3, "white matter", transform=axC.get_xaxis_transform(), ha="center", fontsize=6)
axC.text(2.9, -0.3, "grey matter", transform=axC.get_xaxis_transform(), ha="center", fontsize=6)
axC.set_ylabel("Spp1+ microglia (%)"); axC.set_xlim(-0.5, 3.9); axC.set_ylim(0, None)
axC.set_title("# 3C  Spp1+ microglia per mouse", loc="left", fontsize=7, fontweight="bold")

# ================= D: clustering per mouse (aged)
axD = fig.add_subplot(gs[1, 4:7])
ca = clus[clus.group == "aged"].sort_values("mouse").reset_index(drop=True)
for i, r_ in ca.iterrows():
    axD.plot([i, i], [100 * r_.expected, 100 * r_.observed], color="#cccccc", lw=0.8)
    axD.scatter(i, 100 * r_.expected, s=14, c="#bbbbbb", zorder=3)
    axD.scatter(i, 100 * r_.observed, s=30, marker="D", c=C_POS, zorder=4)
    axD.text(i + 0.2, 100 * r_.observed, sig(r_.p), va="center", fontsize=5.5)
axD.set_xticks(range(len(ca))); axD.set_xticklabels(ca.mouse, fontsize=5.8); axD.set_xlim(-0.6, len(ca) - 0.2)
axD.set_ylabel("Clustered Spp1+ (%)"); axD.set_ylim(0, 100 * ca.observed.max() * 1.3)
axD.scatter([], [], s=24, marker="D", c=C_POS, label="observed"); axD.scatter([], [], s=12, c="#bbbbbb", label="expected")
axD.legend(fontsize=5, loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=2, frameon=False)
axD.set_title("# 3D  Clustering, aged", loc="left", fontsize=7, fontweight="bold")

# ================= E: identity of Spp1+ microglia (aged, depth-matched)
axE = fig.add_subplot(gs[1, 8:12])
GE = [g for g in ["P2ry12", "Tmem119", "Sall1", "Cx3cr1", "Trem2", "Tyrobp", "Apoe", "Cd9", "Clec7a", "Cst7", "Lpl", "Itgax", "Gpnmb", "Cd74"] if g in ident.index]
idf = ident.loc[GE]
yy = np.arange(len(GE))[::-1]
for y_, g in zip(yy, GE):
    c_ = C_POS if idf.loc[g, "diff"] > 0 else "#7f8fb3"
    axE.plot([0, idf.loc[g, "diff"]], [y_, y_], color=c_, lw=1.2); axE.scatter(idf.loc[g, "diff"], y_, s=22, color=c_, zorder=3)
    if idf.loc[g, "q"] < 0.05:
        axE.text(idf.loc[g, "diff"] + (2 if idf.loc[g, "diff"] > 0 else -2), y_, sig(idf.loc[g, "q"]), va="center",
                 ha="left" if idf.loc[g, "diff"] > 0 else "right", fontsize=5)
axE.axvline(0, color="#888888", lw=0.6)
axE.set_yticks(yy); axE.set_yticklabels(GE, style="italic", fontsize=5.8)
lim = idf["diff"].abs().max() * 1.3; axE.set_xlim(-lim, lim)
axE.set_xlabel("Spp1+ minus Spp1− microglia\n(percentage points detecting)", fontsize=6)
axE.set_title("# 3E  Identity, aged", loc="left", fontsize=7, fontweight="bold")

# ================= F: astrocyte neighbours per aged mouse (cell bodies and astrocyte RNA), with 95% CI
axF = fig.add_subplot(gs[2, 0:4])
fr = []
for mo, d in nb.groupby("mouse"):
    if d.spp1.sum() < 10: continue
    f1 = smf.glm("n_astro_25 ~ spp1 + C(ctx_bin)", data=d, family=sm.families.Poisson()).fit(cov_type="HC1")
    f2 = smf.glm("astro_rna ~ spp1 + C(ctx_bin) + n_cells25", data=d, family=sm.families.Poisson()).fit(cov_type="HC1")
    for lab_, f_ in [("astrocyte cell bodies", f1), ("astrocyte RNA", f2)]:
        lo, hi = np.exp(f_.conf_int().loc["spp1"])
        fr.append({"mouse": mo, "measure": lab_, "RR": np.exp(f_.params["spp1"]), "lo": lo, "hi": hi, "p": f_.pvalues["spp1"]})
fr = pd.DataFrame(fr)
ms = sorted(fr.mouse.unique())
for j, (lab_, c_) in enumerate([("astrocyte cell bodies", C_AS), ("astrocyte RNA", "#7f8fb3")]):
    t = fr[fr.measure == lab_].set_index("mouse").loc[ms]
    y_ = np.arange(len(ms)) * 2.2 + j * 0.7
    axF.hlines(y_, np.log2(t.lo), np.log2(t.hi), color=c_, lw=1.0); axF.scatter(np.log2(t.RR), y_, s=20, c=c_, zorder=3, label=lab_)
axF.axvline(0, color="#999999", lw=0.6, ls="--")
axF.set_yticks(np.arange(len(ms)) * 2.2 + 0.35); axF.set_yticklabels([LBLM[m_] for m_ in ms], fontsize=5.8)
axF.invert_yaxis(); axF.set_xlim(-1.5, 1.5)
axF.set_xlabel("log2 rate ratio, Spp1+ vs Spp1−\n(within 25 µm)", fontsize=6)
axF.legend(fontsize=5, loc="upper center", bbox_to_anchor=(0.5, -0.32), ncol=2, frameon=False)
axF.set_title("# 3F  Astrocyte neighbours", loc="left", fontsize=7, fontweight="bold")

# ================= G: reactive neighbouring astrocytes per aged mouse (log2 fold change, stars from shuffles)
axG = fig.add_subplot(gs[2, 5:8])
GG = [g for g in ["Gfap", "Serpina3n", "C4b", "Vim", "Cd44", "Aqp4", "Lgals3", "Aldh1l1", "Slc1a2", "Ntsr2"] if g in react.gene.unique()]
ra = react[react.group == "aged"]
M = ra.pivot_table(index="gene", columns="mouse", values="log2FC").loc[GG, ms]
P = ra.pivot_table(index="gene", columns="mouse", values="p").loc[GG, ms]
im_ = axG.imshow(M.clip(-3, 3).values, cmap=CM_DIV, vmin=-3, vmax=3, aspect="auto")
for i in range(len(GG)):
    for j in range(len(ms)):
        if P.values[i, j] < 0.05: axG.text(j, i, sig(P.values[i, j]), ha="center", va="center", fontsize=5)
axG.set_yticks(range(len(GG))); axG.set_yticklabels(GG, style="italic", fontsize=5.8)
axG.set_xticks(range(len(ms))); axG.set_xticklabels(ms, fontsize=5.8)
axG.axhline(6.5, color="white", lw=1.5)
cb = plt.colorbar(im_, ax=axG, shrink=0.6, pad=0.03); cb.set_label("log2 fold change", fontsize=5.5); cb.ax.tick_params(labelsize=5)
axG.set_title("# 3G  Reactive neighbours", loc="left", fontsize=7, fontweight="bold")

# ================= H: widespread vs local reactivity (Gfap+ astrocytes in white matter)
axH = fig.add_subplot(gs[2, 9:12])
gf = glob[glob.gene == "Gfap"].copy()
gf["slide"] = gf.mouse.map(age_tab.set_index("mouse").slide)
w = gf.pivot(index="slide", columns="group", values="det_far_wm")
wn = gf[gf.group == "aged"].set_index("slide")["det_near"]
for sl in w.index:
    axH.plot([0, 1], [w.loc[sl, "young"], w.loc[sl, "aged"]], color="#bbbbbb", lw=0.7)
    axH.plot([1, 2], [w.loc[sl, "aged"], wn.loc[sl]], color="#e3b3c7", lw=0.7)
axH.scatter(np.zeros(len(w)), w.young, s=18, c=C_YOUNG, zorder=3)
axH.scatter(np.ones(len(w)), w.aged, s=18, c=C_OLD, zorder=3)
axH.scatter(np.full(len(wn), 2), wn, s=18, c=C_POS, zorder=3)
p_far = ttest_rel(w.aged, w.young).pvalue; p_loc = ttest_rel(wn.loc[w.index], w.aged).pvalue
for x0, x1, p_, yb in [(0, 1, p_far, 102), (1, 2, p_loc, 110)]:
    axH.plot([x0, x0, x1, x1], [yb - 2, yb, yb, yb - 2], color="#333333", lw=0.6); axH.text((x0 + x1) / 2, yb + 1, sig(p_), ha="center", fontsize=6)
axH.set_xticks([0, 1, 2]); axH.set_xticklabels(["distant\n4 mo", "distant\naged", "adjacent\naged"], fontsize=5.6)
axH.set_ylabel("Gfap+ astrocytes in\nwhite matter (%)"); axH.set_ylim(0, 118); axH.set_xlim(-0.4, 2.4)
axH.set_title("# 3H  Widespread and local", loc="left", fontsize=7, fontweight="bold")
save(fig, "Fig3")

# ================= Supplementary Table S4: source data of Figure 3
def write_book(path, title, sheets):
    readme = pd.DataFrame([(n, d, len(df)) for n, d, df in sheets], columns=["Sheet", "Content", "Rows"])
    with pd.ExcelWriter(path, engine="openpyxl") as xw:
        readme.to_excel(xw, sheet_name="README", index=False, startrow=2)
        for n, d, df in sheets: df.to_excel(xw, sheet_name=str(n)[:31], index=False, startrow=2)
        wb = xw.book; wb["README"]["A1"] = title; wb["README"]["A1"].font = Font(bold=True, size=12)
        for n, d, df in sheets:
            ws = wb[str(n)[:31]]; ws["A1"] = d; ws["A1"].font = Font(bold=True)
    print("written:", path.name)
XLS = OUT / "final_data" / "supplementary_tables"; XLS.mkdir(parents=True, exist_ok=True)
write_book(XLS / "Supplementary_Table_S4_source_data_Figure3.xlsx", "Supplementary Table S4. Source data of Figure 3 (CosMx cohort, Carver et al. 2026, 4 mo vs 24 to 25 mo)", [
    ("3_mice", "Mice and sections: slide, section, age group (assigned by C4b detection), cells", age_tab.reset_index()),
    ("3C", f"3C: Spp1+ microglia (at least {SPP1_MIN} Spp1 transcripts) per mouse and tissue type", per_mouse),
    ("3D", "3D: clustering, share of Spp1+ microglia with another Spp1+ microglia within 50 µm vs 500 shuffles within tissue type", clus),
    ("3E", "3E: identity, % of microglia detecting each gene after downsampling to 100 transcripts, aged mice pooled, Fisher's exact test, BH", ident.reset_index(names="gene")),
    ("3F", "3F: astrocyte cell bodies and astrocyte RNA within 25 µm, rate ratio Spp1+ vs Spp1-, Poisson regression with tissue type (RNA also with number of cells)", fr),
    ("3F_contact", "3F: share of microglia with an astrocyte within 25 µm, per mouse (cell level)", astro),
    ("3F_niche", "3F: all niche measures per mouse (astrocyte RNA, nearest astrocyte, radii 10 to 50 µm, GFAP protein)", res),
    ("3G", "3G: astrocytes within 25 µm of Spp1+ vs Spp1- microglia (more than 50 µm from Spp1+), log2 fold change and shuffle p per mouse", react),
    ("3H", "3H: detection of reactive genes in astrocytes far from (more than 50 µm) and adjacent to (25 µm) Spp1+ microglia", glob)])

# Figure 3 also to final_data/figures (TIFF for Frontiers)
from PIL import Image
FINAL = OUT / "final_data" / "figures"; FINAL.mkdir(parents=True, exist_ok=True)
for ext in ["pdf", "png"]:
    fig.savefig(FINAL / f"Fig3.{ext}", dpi=300, bbox_inches="tight")
Image.open(FINAL / "Fig3.png").convert("RGB").save(FINAL / "Fig3.tif", compression="tiff_lzw", dpi=(300, 300))